# Attentia Drive — Binary Distraction Classifier Training

Trains MobileNetV2 on the State Farm Distracted Driver Detection dataset.
- **Binary task**: c0 = attentive, c1–c9 = distracted
- **Output**: float16-quantized TFLite model for edge deployment

### Setup steps before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your Kaggle API key (`kaggle.json`) when prompted in Cell 2, **OR** manually upload the dataset zip to your Google Drive
3. Run all cells in order

In [ ]:
# Cell 1 — Check GPU
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPUs:', gpus)
if not gpus:
    print('WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
else:
    print('GPU ready.')

In [ ]:
# Cell 2 — Download State Farm dataset via Kaggle API
# Upload your kaggle.json when prompted (get it from kaggle.com → Account → API)
import os
from google.colab import files

print('Upload your kaggle.json file:')
uploaded = files.upload()

# Install credentials
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Download and unzip dataset (~3.8 GB)
!pip install -q kaggle
!kaggle competitions download -c state-farm-distracted-driver-detection -p /content/
!unzip -q /content/state-farm-distracted-driver-detection.zip -d /content/statefarm/
print('Dataset ready.')

In [ ]:
# Cell 2b — ALTERNATIVE: Mount Google Drive (if you already have the dataset there)
# Skip this cell if you used Cell 2 above.

# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = '/content/drive/MyDrive/statefarm'  # adjust path as needed

In [ ]:
# Cell 3 — Config
from pathlib import Path

DATA_DIR    = Path('/content/statefarm/imgs/train')
BINARY_DIR  = Path('/content/binary')
OUTPUT_DIR  = Path('/content/models')
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_TFLITE = OUTPUT_DIR / 'classifier.tflite'

IMG_SIZE    = (224, 224)
BATCH_SIZE  = 64       # larger batch = faster on GPU
EPOCHS_P1   = 15       # Phase 1: frozen base
EPOCHS_P2   = 10       # Phase 2: fine-tune top layers
LR          = 1e-4
VAL_SPLIT   = 0.20
SEED        = 42

ATTENTIVE_CLASSES  = {'c0'}
DISTRACTED_CLASSES = {'c1','c2','c3','c4','c5','c6','c7','c8','c9'}

print('Config OK')
print('Train data:', DATA_DIR)
print('Classes:', sorted(DATA_DIR.iterdir()) if DATA_DIR.exists() else 'NOT FOUND — check dataset path')

In [ ]:
# Cell 4 — Prepare binary dataset (symlinks to avoid copying ~4 GB)
import os, random
from pathlib import Path

def prepare_binary_dataset():
    for split in ('train', 'val'):
        for label in ('attentive', 'distracted'):
            (BINARY_DIR / split / label).mkdir(parents=True, exist_ok=True)

    random.seed(SEED)
    stats = {'train_attentive': 0, 'train_distracted': 0,
             'val_attentive': 0,   'val_distracted': 0}

    for cls_dir in sorted(DATA_DIR.iterdir()):
        if not cls_dir.is_dir():
            continue
        cls_name = cls_dir.name
        if cls_name in ATTENTIVE_CLASSES:
            label = 'attentive'
        elif cls_name in DISTRACTED_CLASSES:
            label = 'distracted'
        else:
            continue

        images = sorted(cls_dir.glob('*.jpg'))
        random.shuffle(images)
        split_idx = int(len(images) * (1 - VAL_SPLIT))

        for img in images[:split_idx]:
            dst = BINARY_DIR / 'train' / label / f'{cls_name}_{img.name}'
            if not dst.exists():
                os.symlink(img, dst)
            stats[f'train_{label}'] += 1

        for img in images[split_idx:]:
            dst = BINARY_DIR / 'val' / label / f'{cls_name}_{img.name}'
            if not dst.exists():
                os.symlink(img, dst)
            stats[f'val_{label}'] += 1

    print('Binary dataset prepared:')
    for k, v in stats.items():
        print(f'  {k}: {v:,}')
    return stats

stats = prepare_binary_dataset()

In [ ]:
# Cell 5 — Class weights & data loaders
from tensorflow import keras
import tensorflow as tf

# Balanced class weights (1:8 attentive:distracted imbalance)
n_att = stats['train_attentive']
n_dis = stats['train_distracted']
total = n_att + n_dis
class_weights = {
    0: total / (2.0 * n_att),
    1: total / (2.0 * n_dis),
}
print(f'Class weights: attentive={class_weights[0]:.3f}, distracted={class_weights[1]:.3f}')

# Datasets
train_ds = keras.utils.image_dataset_from_directory(
    BINARY_DIR / 'train',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='binary', seed=SEED,
    class_names=['attentive', 'distracted'],
)
val_ds = keras.utils.image_dataset_from_directory(
    BINARY_DIR / 'val',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='binary', seed=SEED,
    class_names=['attentive', 'distracted'],
)

preprocess = keras.applications.mobilenet_v2.preprocess_input

augment = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomBrightness(0.2),
    keras.layers.RandomContrast(0.2),
], name='augmentation')

def prepare_train(img, lbl):
    return augment(preprocess(img), training=True), lbl

def prepare_val(img, lbl):
    return preprocess(img), lbl

train_ds = train_ds.map(prepare_train, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.map(prepare_val,   num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

print('Data loaders ready.')

In [ ]:
# Cell 6 — Build model
from tensorflow import keras

base = keras.applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base.trainable = False

model = keras.Sequential([
    base,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation='sigmoid'),
], name='distraction_classifier')

model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
# Cell 7 — Phase 1: Train classification head (frozen base)
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=3,
        restore_best_weights=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, verbose=1,
    ),
]

print('Phase 1: training classification head...')
history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_P1, callbacks=callbacks,
    class_weight=class_weights,
)

In [ ]:
# Cell 8 — Phase 2: Fine-tune top layers of MobileNetV2
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(LR / 10),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

print('Phase 2: fine-tuning top 30 layers...')
history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_P2, callbacks=callbacks,
    class_weight=class_weights,
)

In [ ]:
# Cell 9 — Evaluate
loss, acc = model.evaluate(val_ds)
print(f'\nFinal validation accuracy: {acc:.1%}  |  loss: {loss:.4f}')

In [ ]:
# Cell 10 — Plot training curves
import matplotlib.pyplot as plt

acc1  = history1.history['accuracy']
val1  = history1.history['val_accuracy']
acc2  = history2.history['accuracy']
val2  = history2.history['val_accuracy']

all_acc = acc1 + acc2
all_val = val1 + val2
epochs  = range(1, len(all_acc) + 1)
p2_start = len(acc1)

plt.figure(figsize=(9, 4))
plt.plot(epochs, all_acc, label='Train accuracy')
plt.plot(epochs, all_val, label='Val accuracy')
plt.axvline(p2_start, color='gray', linestyle='--', label='Phase 2 start')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training History')
plt.legend()
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=120)
plt.show()
print('Saved training_curves.png')

In [ ]:
# Cell 11 — Export to TFLite (float16 quantized)
import numpy as np

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

OUTPUT_TFLITE.write_bytes(tflite_model)
print(f'Saved: {OUTPUT_TFLITE}  ({len(tflite_model)/1e6:.1f} MB)')

# Quick verification
interp = tf.lite.Interpreter(model_path=str(OUTPUT_TFLITE))
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print(f'Input:  shape={inp["shape"]} dtype={inp["dtype"]}')
print(f'Output: shape={out["shape"]} dtype={out["dtype"]}')

test = keras.applications.mobilenet_v2.preprocess_input(
    np.random.rand(1, 224, 224, 3).astype(np.float32)
)
interp.set_tensor(inp['index'], test)
interp.invoke()
result = interp.get_tensor(out['index'])
print(f'Test inference output: {result.flatten()[0]:.4f}')
print('TFLite model verified.')

In [ ]:
# Cell 12 — Download the model
from google.colab import files
files.download(str(OUTPUT_TFLITE))
files.download('/content/training_curves.png')
print('Download started. Place classifier.tflite in your models/ directory.')